# SVM MFCC-10
Trains linear SVM classifiers with grouped 5-fold cross-validation on MFCC-10 data.

In [ ]:
import sys
from pathlib import Path

UTILS_GITHUB_RAW_BASE_URL = "https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/utils"
COMMON_UTILS_FILE = "common_utils.py"
MODEL_UTILS_FILE = "svm_utils.py"
MODEL_DIR = "SVM"
common_dirs = [Path.cwd() / "utils", Path.cwd().parent / "utils", Path("/content/utils"), Path("/content/drive/MyDrive/STUDA/src/utils")]
model_dirs = [Path.cwd(), Path.cwd() / MODEL_DIR, Path.cwd().parent / MODEL_DIR, Path("/content") / MODEL_DIR, Path("/content/drive/MyDrive/STUDA/src") / MODEL_DIR]
common_dir = next((directory for directory in common_dirs if (directory / COMMON_UTILS_FILE).exists()), None)
model_dir = next((directory for directory in model_dirs if (directory / MODEL_UTILS_FILE).exists()), None)

if (common_dir is None or model_dir is None) and UTILS_GITHUB_RAW_BASE_URL:
    import urllib.request
    common_dir = Path("/content/utils")
    model_dir = Path("/content") / MODEL_DIR
    common_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{UTILS_GITHUB_RAW_BASE_URL}/common_utils.py", common_dir / COMMON_UTILS_FILE)
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/Nabuhodonozzor/uuv-detection/main/{MODEL_DIR}/{MODEL_UTILS_FILE}", model_dir / MODEL_UTILS_FILE)

if common_dir is None or model_dir is None:
    raise FileNotFoundError("Required common and model utility files were not found. Sync, upload, mount, or set UTILS_GITHUB_RAW_BASE_URL.")

sys.path.insert(0, str(common_dir))
sys.path.insert(0, str(model_dir))
print(f"Using common utilities from: {common_dir}")
print(f"Using SVM utilities from: {model_dir}")


In [ ]:
import pandas as pd
from IPython.display import display
from google.colab import files

from common_utils import (
    configure_kaggle_access, cross_validate_sklearn_models_for_variants,
    evaluate_models_for_variants, extract_zip, prepare_mfcc_dataset_variants,
    summarize_cross_validation, train_final_sklearn_models_for_variants,
    zip_artifacts,
)
from svm_utils import build_svm_model, save_svm_artifacts


In [ ]:
DATASET_KEY = "mfcc10"
DATASET_LABEL = "MFCC-10"
DATASET_SLUG = "pawedyrda/mfcc10"
ARCHIVE_PATH = Path("/content/mfcc10.zip")


In [ ]:
configure_kaggle_access("Kaggle")
!kaggle datasets download -d {DATASET_SLUG} -p /content --force
extract_zip(ARCHIVE_PATH, "/content")
DATA_PATH = Path("/content") / f"{DATASET_KEY}.npz"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Expected feature archive: {DATA_PATH}")
print(f"Using feature archive: {DATA_PATH}")


In [ ]:
cv_dataset = prepare_mfcc_dataset_variants(DATA_PATH)
final_variants = cv_dataset.final_variants()
print(f"Split ID: {cv_dataset.split_id}")
print(f"CV folds: {cv_dataset.n_splits}")
print(f"MFCC input shape: {cv_dataset.normal.cv_data.shape[1:]}")


In [ ]:
cv_multilabel_results = cross_validate_sklearn_models_for_variants(
    build_svm_model, cv_dataset, "multilabel", DATASET_LABEL,
)
cv_binary_results = cross_validate_sklearn_models_for_variants(
    build_svm_model, cv_dataset, "binary", DATASET_LABEL,
)
cv_results = pd.concat([cv_multilabel_results, cv_binary_results], ignore_index=True)
cv_summary = summarize_cross_validation(cv_results)
display(cv_summary)

multilabel_models = train_final_sklearn_models_for_variants(
    build_svm_model, cv_dataset, "multilabel",
)
binary_models = train_final_sklearn_models_for_variants(
    build_svm_model, cv_dataset, "binary",
)


In [ ]:
multilabel_results = evaluate_models_for_variants(multilabel_models, final_variants, "multilabel", DATASET_LABEL)
binary_results = evaluate_models_for_variants(binary_models, final_variants, "binary", DATASET_LABEL)
comparison_results = pd.concat([
    multilabel_results.assign(task="multilabel"),
    binary_results.assign(task="binary"),
], ignore_index=True)
display(comparison_results[["task", "Model", "precision", "recall", "f1-score", "support"]])


In [ ]:
split_metadata = {
    "split_id": cv_dataset.split_id,
    "n_splits": cv_dataset.n_splits,
    "n_mfcc": cv_dataset.n_mfcc,
}
save_dir = save_svm_artifacts(
    f"/content/saved_artifacts/svm_{DATASET_KEY}",
    DATASET_KEY,
    multilabel_models,
    binary_models,
    multilabel_results,
    binary_results,
    cv_results=cv_results,
    cv_summary=cv_summary,
    split_metadata=split_metadata,
)
comparison_results.to_csv(save_dir / f"svm_comparison_{DATASET_KEY}.csv", index=False)
archive_path = zip_artifacts(save_dir, f"/content/svm_models_and_results_{DATASET_KEY}.zip")
files.download(str(archive_path))
